# Feature Engineering

## K-mer Feature Extraction

**What are k-mers?**
A k-mer is a contiguous subsequence of length k from an amino acid sequence.
For example, the sequence `ACDEF` with k=2 produces: `AC`, `CD`, `DE`, `EF`.

**Why k-mers for protein function prediction?**
- Proteins with similar functions tend to share similar local sequence patterns
- K-mer frequencies capture the amino acid composition and local sequence context
- Fixed-length vectors — regardless of protein length, every protein gets a vector of the same size
- Model-agnostic — the same feature vector is fed to LR, SVM and CNN ensuring a fair comparison

**Settings:**
| Parameter | Value | Description |
|---|---|---|
| `K` | 2 | Length of each k-mer (bi-peptide) |
| `AMINO_ACIDS` | 20 standard AAs | `ACDEFGHIKLMNPQRSTVWY` |
| Feature dimension `d` | 400 | $20^2 = 400$ possible 2-mers |

**How `sequence_to_kmer_vector()` works:**
1. Slides a window of size K across the sequence
2. Counts occurrences of each k-mer
3. Normalises by total k-mer count so vectors are comparable across proteins of different lengths
4. Returns a float32 vector of length 400

**Note:** K=2 was chosen for computational efficiency. K=3 produces richer features (8,000 dimensions) but requires significantly more RAM and training time. All three models (LR, SVM, CNN) use identical k-mer vectors as input, isolating model capacity as the only variable in the 3×3 comparison.

In [ ]:
# Section 4: K-mer Feature Builder 

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
K = 2
all_kmers = ["".join(p) for p in iproduct(AMINO_ACIDS, repeat=K)]
kmer_index = {kmer: i for i, kmer in enumerate(all_kmers)}

def sequence_to_kmer_vector(seq, k=K):
    seq = str(seq).upper()
    vec = np.zeros(len(all_kmers), dtype=np.float32)
    total = 0
    for i in range(len(seq) - k + 1):
        kmer = seq[i:i+k]
        if kmer in kmer_index:
            vec[kmer_index[kmer]] += 1
            total += 1
    if total > 0:
        vec /= total
    return vec